# FastSpeech2 Training on Google Colab

This notebook demonstrates how to train a FastSpeech2 text-to-speech model on Google Colab.

**Features:**
- Automatic GPU detection and setup
- Supports LJSpeech, AISHELL-3, and LibriTTS datasets
- Interactive training with real-time visualization
- Model checkpointing and resumable training
- Inference on custom text
- TensorBoard integration for monitoring

**Note:** This notebook requires a GPU (preferably T4 or higher) for reasonable training times.

## 1. Setup Environment

First, let's check GPU availability and install dependencies.

In [ ]:
import torch
import os

# Check GPU availability
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"PyTorch Version: {torch.__version__}")

In [ ]:
# Install system dependencies
!apt-get update -qq
!apt-get install -y -qq libsndfile1

# Install Python dependencies
requirements = [
    'g2p-en>=2.1.0,<3.0',
    'inflect>=4.1.0,<6.0',
    'librosa>=0.9.0,<1.0.0',
    'matplotlib>=3.4.0,<4.0.0',
    'numba>=0.54.0,<0.60.0',
    'numpy>=1.22.0,<2.0.0',
    'pypinyin>=0.39.0,<0.50.0',
    'pyworld>=0.3.0,<0.4.0',
    'PyYAML>=6.0.0,<7.0.0',
    'scikit-learn>=1.0.0,<2.0.0',
    'scipy>=1.9.0,<2.0.0',
    'soundfile>=0.12.0,<1.0.0',
    'tensorboard>=2.12.0,<3.0.0',
    'tgt==1.4.4',
    'torch>=1.12.0,<3.0.0',
    'tqdm>=4.64.0,<5.0.0',
    'unidecode>=1.3.0,<2.0.0',
    'gdown'
]

for req in requirements:
    !pip install -q "{req}"

print("Dependencies installed successfully!")

## 2. Clone FastSpeech2 Repository

Clone the FastSpeech2 repository or mount it from Google Drive.

In [ ]:
import os
import shutil

# Option 1: Use the repository path (if already cloned/uploaded)
# Option 2: Clone from GitHub

REPO_PATH = "/content/FastSpeech2"

if not os.path.exists(REPO_PATH):
    print("Cloning FastSpeech2 repository...")
    # Replace with your repository URL
    !git clone https://github.com/ming024/FastSpeech2.git {REPO_PATH}
    print("Repository cloned successfully!")
else:
    print(f"Repository already exists at {REPO_PATH}")

os.chdir(REPO_PATH)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# List available datasets and configurations
import os

print("\nAvailable Configurations:")
config_path = os.path.join(REPO_PATH, "config")
for dataset in sorted(os.listdir(config_path)):
    dataset_path = os.path.join(config_path, dataset)
    if os.path.isdir(dataset_path):
        print(f"  - {dataset}")
        for config_file in sorted(os.listdir(dataset_path)):
            print(f"      - {config_file}")

## 3. Dataset Preparation

Download and prepare the LJSpeech dataset (single-speaker English TTS).

**Note:** This fork uses learned alignment during training, so no MFA (Montreal Forced Aligner) is needed.

In [ ]:
# Configuration
DATASET = "LJSpeech"  # Options: "LJSpeech", "AISHELL3", "LibriTTS"
DATASET_PATH = os.path.join(REPO_PATH, "datasets", DATASET)
PREPROCESSED_PATH = os.path.join(REPO_PATH, "preprocessed_data", DATASET)

os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(PREPROCESSED_PATH, exist_ok=True)

print(f"Dataset: {DATASET}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Preprocessed path: {PREPROCESSED_PATH}")

In [ ]:
# Verify dataset structure
ljspeech_path = os.path.join(DATASET_PATH, "LJSpeech-1.1")
if os.path.exists(ljspeech_path):
    wavs_path = os.path.join(ljspeech_path, "wavs")
    metadata_path = os.path.join(ljspeech_path, "metadata.csv")
    
    num_wavs = len([f for f in os.listdir(wavs_path) if f.endswith('.wav')]) if os.path.exists(wavs_path) else 0
    print(f"Dataset structure verified:")
    print(f"  Audio files: {num_wavs}")
    print(f"  Metadata: {os.path.exists(metadata_path)}")
    
    # Display first 3 metadata lines
    if os.path.exists(metadata_path):
        with open(metadata_path, 'r') as f:
            print(f"\nFirst 3 entries:")
            for i, line in enumerate(f):
                if i < 3:
                    parts = line.strip().split('|')
                    print(f"  {parts[0]}: {parts[2][:60]}...") 
                else:
                    break
else:
    print("LJSpeech dataset not found. Please check download.")

## 4. Preprocessing

Extract mel-spectrograms, pitch, and energy features from the raw audio.

**Note:** Alignment durations are learned dynamically during training, so no external alignment extraction is needed.

In [ ]:
# Update preprocessing config with Colab paths
import yaml

config_path = os.path.join(REPO_PATH, f"config/{DATASET}/preprocess.yaml")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update paths for Colab
config['path']['corpus_path'] = os.path.join(DATASET_PATH, "LJSpeech-1.1")
config['path']['raw_path'] = os.path.join(REPO_PATH, "raw_data", DATASET)
config['path']['preprocessed_path'] = os.path.join(REPO_PATH, "preprocessed_data", DATASET)

# Create raw_data directory
os.makedirs(config['path']['raw_path'], exist_ok=True)

# Write updated config
with open(config_path, 'w') as f:
    yaml.dump(config, f)

print("Updated preprocessing config:")
print(f"  corpus_path: {config['path']['corpus_path']}")
print(f"  raw_path: {config['path']['raw_path']}")
print(f"  preprocessed_path: {config['path']['preprocessed_path']}")

In [ ]:
import subprocess
import sys

os.chdir(REPO_PATH)

# Step 1: Prepare dataset (extract and normalize audio)
config_path = f"config/{DATASET}/preprocess.yaml"

print(f"Running prepare_align with config: {config_path}")
print("This extracts and normalizes audio from the dataset...\n")
result = subprocess.run(
    [sys.executable, "prepare_align.py", config_path],
    capture_output=True,
    text=True,
    timeout=600
)

print(result.stdout)
if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)
print(f"\nReturn code: {result.returncode}")

if result.returncode == 0:
    print("✓ Dataset preparation completed successfully!")
else:
    print("✗ Error during preparation")

In [ ]:
# Step 2: Extract features (mel-spectrogram, pitch, energy)
print(f"Extracting features from {DATASET} dataset...")
print("This extracts mel-spectrograms, pitch, and energy features...\n")
result = subprocess.run(
    [sys.executable, "preprocess.py", config_path],
    capture_output=True,
    text=True,
    timeout=3600  # 1 hour timeout
)

print("Output (last 1500 chars):")
print(result.stdout[-1500:] if len(result.stdout) > 1500 else result.stdout)
if result.stderr:
    print("\nErrors (last 1000 chars):")
    print(result.stderr[-1000:] if len(result.stderr) > 1000 else result.stderr)
print(f"\nReturn code: {result.returncode}")

if result.returncode == 0:
    print("✓ Feature extraction completed successfully!")
else:
    print("✗ Error during feature extraction")

## 5. Model Training

Configure and start training the FastSpeech2 model with learned alignment.

In [ ]:
# Verify preprocessing results
import json

preprocessed_path = os.path.join(REPO_PATH, "preprocessed_data", DATASET)

print(f"Preprocessed data at: {preprocessed_path}")
print(f"\nContents:")
if os.path.exists(preprocessed_path):
    for item in sorted(os.listdir(preprocessed_path)):
        item_path = os.path.join(preprocessed_path, item)
        if os.path.isfile(item_path):
            size = os.path.getsize(item_path) / (1024**2)  # Convert to MB
            print(f"  {item}: {size:.2f} MB")
        elif os.path.isdir(item_path):
            num_files = len(os.listdir(item_path))
            total_size = sum(os.path.getsize(os.path.join(item_path, f)) for f in os.listdir(item_path)) / (1024**2)
            print(f"  {item}/: {num_files} files ({total_size:.2f} MB)")

# Load and display statistics
stats_file = os.path.join(preprocessed_path, "stats.json")
if os.path.exists(stats_file):
    with open(stats_file, 'r') as f:
        stats = json.load(f)
    print(f"\nDataset Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value}")
    print("\n✓ Preprocessing verification complete!")
else:
    print("\nWarning: stats.json not found. Check if preprocessing completed successfully.")

In [ ]:
# Start TensorBoard
log_path = os.path.join(REPO_PATH, "output", "log", DATASET)
os.makedirs(log_path, exist_ok=True)

print(f"TensorBoard log path: {log_path}")
print("\nStarting TensorBoard in the background...")

%load_ext tensorboard
%tensorboard --logdir {log_path}

In [ ]:
import subprocess
import sys

os.chdir(REPO_PATH)

# Build training command
train_command = [
    sys.executable,
    "train.py",
    "-p", f"config/{DATASET}/preprocess.yaml",
    "-m", f"config/{DATASET}/model.yaml",
    "-t", f"config/{DATASET}/train.yaml",
    "-r", str(TRAINING_CONFIG["restore_step"]),
]

print(f"Training command: {' '.join(train_command)}")
print("\nStarting training...\n")

# Start training
result = subprocess.run(
    train_command,
    capture_output=False,  # Show live output
    text=True
)

print(f"\nTraining completed with return code: {result.returncode}")

## 6. Inference

Use the trained model to synthesize speech from text.

In [ ]:
# Training configuration
TRAINING_CONFIG = {
    "dataset": DATASET,
    "restore_step": 0,  # Change this to resume from a checkpoint
    "batch_size": 16,  # Reduce if OOM errors occur
    "epochs": 100,
    "learning_rate": 0.001,
    "log_interval": 100,  # Log every N steps
    "save_interval": 1000,  # Save checkpoint every N steps
    "val_interval": 1000,  # Validate every N steps
}

print("Training Configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Start TensorBoard
log_path = os.path.join(REPO_PATH, "output", "log", DATASET)
os.makedirs(log_path, exist_ok=True)

print(f"TensorBoard log path: {log_path}")
print("\nStarting TensorBoard in the background...")

%load_ext tensorboard
%tensorboard --logdir {log_path}

In [ ]:
import subprocess
import sys

os.chdir(REPO_PATH)

# Build training command
train_command = [
    sys.executable,
    "train.py",
    "-p", f"config/{DATASET}/preprocess.yaml",
    "-m", f"config/{DATASET}/model.yaml",
    "-t", f"config/{DATASET}/train.yaml",
    "-r", str(TRAINING_CONFIG["restore_step"]),
]

print(f"Training command: {' '.join(train_command)}")
print("\nStarting training...\n")

# Start training
result = subprocess.run(
    train_command,
    capture_output=False,  # Show live output
    text=True
)

print(f"\nTraining completed with return code: {result.returncode}")

In [ ]:
# Visualize mel-spectrogram
import matplotlib.pyplot as plt
import numpy as np

# Look for any generated figures
import os
from pathlib import Path

result_path = os.path.join(REPO_PATH, "output", "result", DATASET)

# Find PNG files (mel-spectrograms)
png_files = list(Path(result_path).glob('*.png'))

if png_files:
    latest_png = sorted(png_files)[-1]
    print(f"Displaying: {latest_png.name}")
    img = plt.imread(str(latest_png))
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No visualization files found.")

## 6. Inference

Use the trained model to synthesize speech from text.

In [ ]:
# Find the latest checkpoint
import os
import re

ckpt_path = os.path.join(REPO_PATH, "output", "ckpt", DATASET)
os.makedirs(ckpt_path, exist_ok=True)

if os.path.exists(ckpt_path):
    checkpoints = [f for f in os.listdir(ckpt_path) if f.endswith('.pth')]
    if checkpoints:
        # Extract step number from checkpoint name
        def get_step(ckpt_name):
            match = re.search(r'\d+', ckpt_name)
            return int(match.group()) if match else 0
        
        latest_ckpt = max(checkpoints, key=get_step)
        restore_step = get_step(latest_ckpt)
        print(f"Found checkpoint: {latest_ckpt}")
        print(f"Restore step: {restore_step}")
    else:
        print("No checkpoints found. Please train the model first.")
        restore_step = 0
else:
    print(f"Checkpoint directory does not exist: {ckpt_path}")
    restore_step = 0

## 7. Advanced Features

Control pitch, energy, and duration of synthesized speech.

In [ ]:
# List all output files
import os
from pathlib import Path

output_path = os.path.join(REPO_PATH, "output")

print("Output Structure:")
for root, dirs, files in os.walk(output_path):
    level = root.replace(output_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    
    subindent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # Limit to first 5 files per directory
        file_size = os.path.getsize(os.path.join(root, file)) / (1024**2)  # MB
        print(f"{subindent}{file} ({file_size:.2f} MB)")
    
    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

In [ ]:
# Create a zip file of the trained model
import shutil
import os

ckpt_path = os.path.join(REPO_PATH, "output", "ckpt", DATASET)
zip_name = f"fastspeech2_{DATASET}_model"
zip_path = f"/content/{zip_name}"

if os.path.exists(ckpt_path) and os.listdir(ckpt_path):
    print(f"Creating zip file: {zip_name}.zip")
    shutil.make_archive(zip_path, 'zip', ckpt_path)
    
    zip_file_path = f"{zip_path}.zip"
    size_mb = os.path.getsize(zip_file_path) / (1024**2)
    print(f"Created: {zip_file_path} ({size_mb:.2f} MB)")
    print(f"\nTo download, click on the file in the file browser or use:")
    print(f"from google.colab import files")
    print(f"files.download('{zip_file_path}')")
else:
    print("No checkpoint files found to download.")

In [ ]:
# Download synthesized audio results
from google.colab import files
import os

result_path = os.path.join(REPO_PATH, "output", "result", DATASET)

if os.path.exists(result_path):
    audio_files = [f for f in os.listdir(result_path) if f.endswith('.wav')]
    
    if audio_files:
        print(f"Found {len(audio_files)} audio files:")
        for audio_file in audio_files[:5]:  # Show first 5
            print(f"  - {audio_file}")
        
        # Create zip of results
        zip_name = f"fastspeech2_{DATASET}_results"
        zip_path = f"/content/{zip_name}"
        print(f"\nCreating zip file: {zip_name}.zip")
        shutil.make_archive(zip_path, 'zip', result_path)
        
        print(f"To download: files.download('{zip_path}.zip')")
    else:
        print("No audio files found in results directory.")
else:
    print(f"Result directory does not exist: {result_path}")

## 9. Tips and Troubleshooting

### Performance Optimization
- **Reduce batch size** if you encounter CUDA out-of-memory errors (in train.yaml)
- **Use mixed precision** training for faster convergence
- **Enable gradient accumulation** for larger effective batch sizes

### Training with Learned Alignment
- This implementation learns alignment automatically during training using the AlignmentNetwork
- No pre-computed alignments (TextGrids/MFA) are required
- Duration values are computed dynamically from the alignment

### Common Issues
1. **CUDA Out of Memory**: Reduce batch_size in config or use smaller model
2. **Dataset not found**: Ensure the dataset is properly extracted and prepare_align was run
3. **Training not progressing**: Check TensorBoard for loss curves
4. **Missing duration files**: Duration is learned during training, no external alignment needed

### Model Improvement
- Train for more steps (adjust total_step in train.yaml)
- Use a larger dataset (LibriTTS for better generalization)
- Tune hyperparameters in the config files

### Resuming Training
- Set `restore_step` to the checkpoint step number
- The model will continue from that checkpoint

## 10. Useful Links

- [FastSpeech2 Paper](https://arxiv.org/abs/2006.04558)
- [GitHub Repository](https://github.com/ming024/FastSpeech2)
- [Audio Samples](https://ming024.github.io/FastSpeech2/)
- [LJSpeech Dataset](https://keithito.com/LJ-Speech-Dataset/)
- [HiFi-GAN Vocoder](https://github.com/jik876/hifi-gan)
- [Alignment Network Paper](https://github.com/idiap/coqui-ai-TTS)